# Book Markdown RAG Pipeline
### Markdown Parsing → BERTTopic → Structure-Aware Chunking → e5-base-v2 Embeddings → FAISS RAG

**Pipeline steps**
1. Mount Google Drive and load `.md` files from the `bookmd` directory  
2. Parse Markdown hierarchy (H1 chapter / H2 section / H3 subsection) — strip images & link URLs  
3. Run **BERTTopic** on section + subsection texts → identify salient topics  
4. Save chapter/section/subsection structure + topics to **JSON**  
5. Structure-aware chunking with 15 % overlap (adapted from the MDA pipeline)  
6. Generate **e5-base-v2** embeddings (`passage:` / `query:` instruction prefix)  
7. Build a **FAISS** vector index (cosine / inner-product on L2-normalised vectors)  
8. **RAG query** demo — retrieve relevant chunks and display context  


## Step 0 — Install dependencies

In [1]:
# Run once per Colab session
import subprocess, sys

PACKAGES = [
    "sentence-transformers",
    "bertopic",
    "umap-learn",
    "hdbscan",
    "faiss-cpu",        # use faiss-gpu if you have a GPU runtime
    "numpy",
    "pandas",
]
for pkg in PACKAGES:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

print("All packages installed.")


All packages installed.


In [10]:
from bertopic import BERTopic

## Step 1 — Mount Google Drive & locate `.md` files

In [5]:
import os
from pathlib import Path
IN_COLAB = True
# ── Option A: Google Colab — mount Drive ─────────────────────────────────

try:
    from google.colab import drive
    drive.mount("/content/drive")
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    print("Not running in Colab — using local path.")

# ── Configuration ─────────────────────────────────────────────────────────
# Set BOOKMD_DIR to wherever your .md files live.
# Examples:
#   Colab + Drive : "/content/drive/MyDrive/bookmd"
#   Colab upload  : "/content/bookmd"
#   Local         : "/path/to/bookmd"
BOOKMD_DIR = "/content/drive/MyDrive/book_rag/bookmd"   # <-- change if needed

# Output paths (written to /content/ in Colab or cwd locally)
OUT_DIR              = Path("/content/drive/MyDrive/book_rag/out") if IN_COLAB else Path(".")
STRUCTURE_JSON       = OUT_DIR / "book_structure_topics.json"
CHUNKS_JSON          = OUT_DIR / "book_chunks_embedded.json"
FAISS_INDEX_PATH     = str(OUT_DIR / "book_faiss.index")
CHUNK_META_JSON      = OUT_DIR / "book_chunk_meta.json"

# Chunking / embedding hyper-params
CHUNK_SIZE_TOKENS    = 128      # ~512 chars per chunk
OVERLAP_PERCENT      = 15
EMBED_MODEL          = "intfloat/e5-base-v2"   # dim = 768
EMBED_BATCH          = 32
TOPIC_MIN_TOPIC_SIZE = 2        # small corpus: allow small clusters

md_files = sorted(Path(BOOKMD_DIR).glob("*.md"))
print(f"Found {len(md_files)} .md file(s):")
for f in md_files:
    print(f"  {f.name}")


Mounted at /content/drive
Found 4 .md file(s):
  ch01-trade-offs-in-data-systems-architecture.md
  ch02_nonfunctional_requirements.md
  ch03_data_models.md
  ch04_storage_and_retrieval.md


## Step 2 — Parse Markdown into hierarchical structure

In [6]:
import re
from typing import List, Dict, Optional, Tuple

# ── Text cleaning helpers ──────────────────────────────────────────────────
def clean_markdown_text(text: str) -> str:
    """Remove images, resolve links to plain text, strip HTML."""
    # Remove images:  ![alt](url)  or  ![alt](url "title")
    text = re.sub(r'!\[([^\]]*)\]\([^)]*\)', '', text)
    # Resolve links: [text](url) -> text
    text = re.sub(r'\[([^\]]+)\]\([^)]+\)', r'\1', text)
    # Strip inline HTML tags
    text = re.sub(r'<[^>]+>', '', text)
    # Strip blockquote markers (keep content)
    text = re.sub(r'^>\s?', '', text, flags=re.MULTILINE)
    # Collapse multiple blank lines
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text.strip()


def parse_markdown_file(filepath: Path) -> Dict:
    """
    Parse a single .md file into a hierarchical structure dict:
        {
          'filename': str,
          'chapter':  str | None,   # H1 title
          'sections': [
            {
              'heading':    str,         # H2
              'intro_text': str,         # text before first H3
              'subsections': [
                {
                  'heading': str,        # H3 (or H4 grouped under parent H3)
                  'text':    str,
                }
              ],
            }
          ],
        }
    """
    raw = filepath.read_text(encoding='utf-8')
    text = clean_markdown_text(raw)
    lines = text.split('\n')

    structure: Dict = {
        'filename':  filepath.name,
        'filepath':  str(filepath),
        'chapter':   None,
        'sections':  [],
    }

    cur_section:    Optional[Dict] = None
    cur_subsection: Optional[Dict] = None
    buf:            List[str]      = []

    def _flush_buf():
        return '\n'.join(buf).strip()

    def _save_buf_to(target: Optional[Dict], key: str):
        nonlocal buf
        if target is not None:
            content = _flush_buf()
            if content:
                # append rather than overwrite if key already has content
                existing = target.get(key, '')
                target[key] = (existing + '\n\n' + content).strip() if existing else content
        buf = []

    for line in lines:
        h1 = re.match(r'^#\s+(.+)$',   line)
        h2 = re.match(r'^##\s+(.+)$',  line)
        h3 = re.match(r'^###\s+(.+)$', line)
        h4 = re.match(r'^####\s+(.+)$', line)   # treat H4 as extra subsection detail

        if h1:
            _save_buf_to(cur_subsection, 'text') if cur_subsection else _save_buf_to(cur_section, 'intro_text')
            structure['chapter'] = h1.group(1).strip()
            cur_section = cur_subsection = None

        elif h2:
            _save_buf_to(cur_subsection, 'text') if cur_subsection else _save_buf_to(cur_section, 'intro_text')
            cur_subsection = None
            cur_section = {
                'heading':     h2.group(1).strip(),
                'intro_text':  '',
                'subsections': [],
            }
            structure['sections'].append(cur_section)

        elif h3:
            _save_buf_to(cur_subsection, 'text') if cur_subsection else _save_buf_to(cur_section, 'intro_text')
            cur_subsection = {
                'heading': h3.group(1).strip(),
                'text':    '',
            }
            if cur_section is not None:
                cur_section['subsections'].append(cur_subsection)

        elif h4:
            # Fold H4 text into the current H3 subsection (or treat as new subsection)
            _save_buf_to(cur_subsection, 'text')
            new_sub = {
                'heading': h4.group(1).strip(),
                'text':    '',
                'level':   4,
            }
            if cur_section is not None:
                cur_section['subsections'].append(new_sub)
            cur_subsection = new_sub

        else:
            buf.append(line)

    # flush remainder
    if cur_subsection:
        _save_buf_to(cur_subsection, 'text')
    elif cur_section:
        _save_buf_to(cur_section, 'intro_text')

    return structure


# ── Parse all files ───────────────────────────────────────────────────────
all_structures: List[Dict] = []
for f in md_files:
    s = parse_markdown_file(f)
    all_structures.append(s)
    n_sec   = len(s['sections'])
    n_sub   = sum(len(sec['subsections']) for sec in s['sections'])
    print(f"  {f.name:55s}  chapter={bool(s['chapter'])}  sections={n_sec:3d}  subsections={n_sub:3d}")

total_sections    = sum(len(s['sections']) for s in all_structures)
total_subsections = sum(len(sec['subsections']) for s in all_structures for sec in s['sections'])
print(f"\nTotal: {total_sections} sections, {total_subsections} subsections across {len(all_structures)} file(s)")


  ch01-trade-offs-in-data-systems-architecture.md          chapter=True  sections=  7  subsections= 13
  ch02_nonfunctional_requirements.md                       chapter=True  sections=  7  subsections= 18
  ch03_data_models.md                                      chapter=True  sections= 14  subsections= 38
  ch04_storage_and_retrieval.md                            chapter=True  sections= 15  subsections= 18

Total: 43 sections, 87 subsections across 4 file(s)


## Step 3 — Topic modelling with BERTTopic

In [11]:
from bertopic import BERTopic

# ── Collect topic-modelling documents ────────────────────────────────────
topic_docs:  List[str]  = []
topic_meta:  List[Dict] = []

MIN_WORDS = 15   # skip very short texts

for s in all_structures:
    chap = s.get('chapter') or s['filename']
    for sec in s['sections']:
        sec_text = (sec.get('intro_text') or '').strip()
        if len(sec_text.split()) >= MIN_WORDS:
            topic_docs.append(sec_text)
            topic_meta.append({
                'type':     'section',
                'chapter':  chap,
                'section':  sec['heading'],
                'subsection': None,
                'source':   s['filename'],
            })
        for sub in sec.get('subsections', []):
            sub_text = (sub.get('text') or '').strip()
            if len(sub_text.split()) >= MIN_WORDS:
                topic_docs.append(sub_text)
                topic_meta.append({
                    'type':       'subsection',
                    'chapter':    chap,
                    'section':    sec['heading'],
                    'subsection': sub['heading'],
                    'source':     s['filename'],
                })

print(f"Topic-modelling corpus: {len(topic_docs)} documents "
      f"({sum(1 for m in topic_meta if m['type']=='section')} sections, "
      f"{sum(1 for m in topic_meta if m['type']=='subsection')} subsections)")


Topic-modelling corpus: 118 documents (35 sections, 83 subsections)


In [ ]:
#improvements ?? some possible but not the option to call llm via api
from bertopic.vectorizers import ClassTfidfTransformer
from sklearn.feature_extraction.text import CountVectorizer

# Domain stopwords on top of English defaults
EXTRA_STOPS = [
    'et', 'al', 'figure', 'example', 'https', 'www', 'com', 'org',
    'chapter', 'section', 'table', 'the', 'id', 'db_set', 'idaho',
    'attractions',  # leaked from demo data
]

vectorizer = CountVectorizer(
    stop_words='english',          # base English list
    ngram_range=(1, 2),            # bigrams → "b_tree", "response_time"
    min_df=2,                      # drop hapax legomena
    max_df=0.85,                   # drop near-universal terms
    token_pattern=r"(?u)\b[a-zA-Z][a-zA-Z_\-]{2,}\b",  # no bare digits
)

ctfidf = ClassTfidfTransformer(reduce_frequent_words=True)  # penalise frequent words further
#how to use with existing better embedding model ? vs LLM not an option
"""
Use LLM-based label generation (BERTopic built-in)
BERTopic ≥0.15 ships representation_model that replaces the c-TF-IDF label with an LLM summary of the representative docs:
"""
from bertopic.representation import KeyBERTInspired, OpenAI, TextGeneration
import openai

# Option A — KeyBERT (no API cost, good for technical text)
from bertopic.representation import KeyBERTInspired
representation_model = KeyBERTInspired()

# Option B — LLM prompt (best quality, uses Claude/OpenAI)
from bertopic.representation import OpenAI
prompt = """
I have a topic from a database systems textbook with these
representative passages:
[DOCUMENTS]
The topic contains these keywords: [KEYWORDS]
Give a precise 3-5 word technical label. Examples:
"B-tree index storage", "Graph query languages", "Fault tolerance design"
Topic label:"""

representation_model = OpenAI(
    client=openai.OpenAI(),
    model="gpt-4o-mini",
    prompt=prompt,
    nr_docs=3,
    delay_in_seconds=2,
)

topic_model = BERTopic(representation_model=representation_model)
"""
The single highest-ROI fix is ngram vectorizer + LLM representation model — those two changes 
alone will transform your labels from stopword soup to meaningful technical phrases without touching 
the embedding model or clustering parameters.
"""

In [16]:
# ── Fit BERTTopic ────────────────────────────────────────────────────────
# We use the default sentence-transformer embedding model inside BERTTopic
# for topic modelling (all-MiniLM-L6-v2 — fast & lightweight).
# The e5-base-v2 model is used exclusively for the RAG embeddings later.

topic_model = BERTopic(
    embedding_model  = "all-MiniLM-L6-v2",
    min_topic_size   = TOPIC_MIN_TOPIC_SIZE,
    nr_topics        =  25, #"auto",
    calculate_probabilities = False,
    verbose          = True,
)

topics, _ = topic_model.fit_transform(topic_docs)

# Topic info
topic_info = topic_model.get_topic_info()
print("\nDiscovered topics (excluding outlier topic -1):")
print(topic_info[topic_info.Name != "-1_"].to_string(index=False))

# Attach topic labels to metadata
for i, (meta, topic_id) in enumerate(zip(topic_meta, topics)):
    topic_words = topic_model.get_topic(topic_id)
    meta['topic_id']    = int(topic_id)
    meta['topic_label'] = ", ".join(w for w, _ in topic_words[:5]) if topic_words else "outlier"

print(f"\nAssigned topics to {len(topic_meta)} documents.")


2026-05-24 15:56:56,333 - BERTopic - Embedding - Transforming documents to embeddings.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/4 [00:00<?, ?it/s]

2026-05-24 15:56:58,829 - BERTopic - Embedding - Completed ✓
2026-05-24 15:56:58,830 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-05-24 15:56:58,965 - BERTopic - Dimensionality - Completed ✓
2026-05-24 15:56:58,966 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-05-24 15:56:58,973 - BERTopic - Cluster - Completed ✓
2026-05-24 15:56:58,974 - BERTopic - Representation - Extracting topics using c-TF-IDF for topic reduction.
2026-05-24 15:56:59,022 - BERTopic - Representation - Completed ✓
2026-05-24 15:56:59,023 - BERTopic - Topic reduction - Reducing number of topics
2026-05-24 15:56:59,023 - BERTopic - Topic reduction - Number of topics (25) is equal or higher than the clustered topics(15).
2026-05-24 15:56:59,024 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-05-24 15:56:59,181 - BERTopic - Representation - Completed ✓



Discovered topics (excluding outlier topic -1):
 Topic  Count                                  Name                                                                  Representation                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                    

In [ ]:
#improvements
from hdbscan import HDBSCAN
hdbscan_model = HDBSCAN(
    min_cluster_size=2,    # default is 10 — too large for 87 docs
    min_samples=1,
    prediction_data=True,
)
topic_model = BERTopic(hdbscan_model=hdbscan_model)

# Option A — better general technical model
from sentence_transformers import SentenceTransformer
embedding_model = SentenceTransformer("BAAI/bge-base-en-v1.5")

# Option B — use the section headings as prefix (E5 style)
# E5 models respond to "query: " / "passage: " prefixes
docs_with_prefix = [f"passage: {doc}" for doc in docs]
# Your subsections all have topic: n/a. After rerunning, write back:
# After fitting: topics, probs = topic_model.fit_transform(docs)
topic_info = topic_model.get_topic_info()
label_map = dict(zip(topic_info['Topic'], topic_info['Name']))

# Match docs back to subsections by index
doc_idx = 0
for book in data['books']:
    for section in book['sections']:
        for subsection in section.get('subsections', []):
            subsection['topic'] = int(topics[doc_idx])
            subsection['topic_name'] = label_map.get(topics[doc_idx], '')
            subsection['topic_prob'] = float(probs[doc_idx].max())
            doc_idx += 1

with open('book_structure_topics_v2.json', 'w') as f:
    json.dump(data, f, indent=2)

## Step 4 — Save structure + topics to JSON

In [14]:
import json

# ── Attach topic info back into the structure dicts ───────────────────────
# Build lookup: (chapter, section, subsection) -> topic info
_lookup: Dict[Tuple, Dict] = {}
for meta in topic_meta:
    key = (meta['chapter'], meta['section'], meta.get('subsection'))
    _lookup[key] = {
        'topic_id':    meta['topic_id'],
        'topic_label': meta['topic_label'],
    }

for s in all_structures:
    chap = s.get('chapter') or s['filename']
    for sec in s['sections']:
        t = _lookup.get((chap, sec['heading'], None), {})
        sec['topic_id']    = t.get('topic_id', -1)
        sec['topic_label'] = t.get('topic_label', 'outlier')
        for sub in sec.get('subsections', []):
            t2 = _lookup.get((chap, sec['heading'], sub['heading']), {})
            sub['topic_id']    = t2.get('topic_id', -1)
            sub['topic_label'] = t2.get('topic_label', 'outlier')

# ── Write STRUCTURE_JSON ──────────────────────────────────────────────────
export = {
    'meta': {
        'n_files':       len(all_structures),
        'n_sections':    total_sections,
        'n_subsections': total_subsections,
        'n_topics':      int(topic_info[topic_info.Name != "-1_"].shape[0]),
        'embed_model':   EMBED_MODEL,
    },
    'topic_summary': topic_info.to_dict(orient='records'),
    'books': all_structures,
}

with open(STRUCTURE_JSON, 'w', encoding='utf-8') as fh:
    json.dump(export, fh, ensure_ascii=False, indent=2, default=str)

size_kb = STRUCTURE_JSON.stat().st_size / 1024
print(f"Saved structure + topics → {STRUCTURE_JSON}  ({size_kb:.1f} KB)")
print(f"  {len(all_structures)} files / {total_sections} sections / {total_subsections} subsections")


Saved structure + topics → /content/drive/MyDrive/book_rag/out/book_structure_topics.json  (396.6 KB)
  4 files / 43 sections / 87 subsections


In [ ]:
"""
Topic 12 (12_https_org_com_cloud) is entirely citation artifacts — URLs and et al patterns. Strip these from chunk text before embedding:
python
"""
import re

def clean_chunk(text):
    text = re.sub(r'https?://\S+', '', text)           # URLs
    text = re.sub(r'\bet\s+al\.?', '', text)           # citations
    text = re.sub(r'\[\d+\]', '', text)                # ref numbers
    text = re.sub(r'Figure\s+\d+[-\.\d]*', '', text)  # figure refs
    text = re.sub(r'Example\s+\d+[-\.\d]*', '', text) # example refs
    return text.strip()




Expected Label Quality After Fixes
![image.png](cite_stop_word_improv.jpg)

## Step 5 — Structure-aware chunking

Adapted directly from the `mda_pipeline` `TextChunker`:  
- **H2 sections** and **H3 subsections** are treated as natural chunk boundaries.  
- Each section body that exceeds `CHUNK_SIZE_TOKENS * 4` chars is split into  
  overlapping sub-chunks with `OVERLAP_PERCENT` % overlap.  
- Falls back to fixed-size chunking when no headers are detected.


In [ ]:
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Tuple

@dataclass
class MDChunk:
    chunk_id:           str
    source_document_id: str   # filename stem
    text:               str
    chunk_index:        int
    total_chunks:       int
    start_char:         int
    end_char:           int
    section_name:       str   # H2 heading  (empty for fixed-size fallback)
    subsection_name:    str   # H3 heading  (may be empty)
    chapter:            str
    source_file:        str
    topic_id:           int   = -1
    topic_label:        str   = ""
    metadata:           Dict  = field(default_factory=dict)


class MarkdownChunker:
    """
    Structure-aware chunker for book-style Markdown files.

    Chunk boundaries follow H2 sections / H3 subsections.
    Large sections are split with percentage overlap, respecting
    sentence / line boundaries (identical logic to MDA TextChunker).
    """

    def __init__(self, chunk_size_tokens: int = 128, overlap_percent: int = 15):
        self.chunk_size   = chunk_size_tokens * 4    # chars (~4 chars/token)
        self.overlap      = int((overlap_percent / 100.0) * self.chunk_size)
        self._h2_re       = re.compile(r'^##\s+(.+)$',  re.MULTILINE)
        self._h3_re       = re.compile(r'^###\s+(.+)$', re.MULTILINE)

    # ── helpers ─────────────────────────────────────────────────────────
    def _find_md_sections(self, text: str) -> List[Tuple[int, str, str]]:
        """Return list of (start_offset, level, heading) for H2/H3 headers."""
        hits: List[Tuple[int, str, str]] = []
        for m in re.finditer(r'^(#{2,3})\s+(.+)$', text, re.MULTILINE):
            level   = 'h2' if len(m.group(1)) == 2 else 'h3'
            heading = m.group(2).strip()
            hits.append((m.start(), level, heading))
        return hits

    def _split_with_overlap(
        self, text: str, source_id: str, meta: Dict,
        section_name: str, subsection_name: str,
        chapter: str, source_file: str,
        topic_id: int, topic_label: str,
        global_idx: int, offset: int,
    ) -> List[MDChunk]:
        out:   List[MDChunk] = []
        start, n = 0, len(text)
        idx   = global_idx

        while start < n:
            end = min(start + self.chunk_size, n)
            if end < n:
                half = start + self.chunk_size // 2
                bp   = max(
                    text.rfind('. ', half, end),
                    text.rfind('\n',  half, end),
                    text.rfind(' ',   half, end),
                )
                if bp > start:
                    end = bp + 1

            piece = text[start:end].strip()
            if piece:
                out.append(MDChunk(
                    chunk_id=f"{source_id}_chunk_{idx}",
                    source_document_id=source_id,
                    text=piece,
                    chunk_index=idx,
                    total_chunks=1,
                    start_char=offset + start,
                    end_char=offset + end,
                    section_name=section_name,
                    subsection_name=subsection_name,
                    chapter=chapter,
                    source_file=source_file,
                    topic_id=topic_id,
                    topic_label=topic_label,
                    metadata=dict(meta),
                ))
                idx += 1

            if end >= n:
                break
            start = max(end - self.overlap, start + 1)

        return out

    # ── main entry ───────────────────────────────────────────────────────
    def chunk_structure(self, parsed: Dict) -> List[MDChunk]:
        """
        Chunk one parsed-structure dict (output of parse_markdown_file).
        Uses H2/H3 boundaries; falls back to fixed-size if none found.
        """
        source_id   = Path(parsed['filepath']).stem
        source_file = parsed['filename']
        chapter     = parsed.get('chapter') or source_id
        all_chunks: List[MDChunk] = []
        global_idx = 0

        for sec in parsed['sections']:
            sec_heading    = sec['heading']
            sec_topic_id   = sec.get('topic_id',    -1)
            sec_topic_lbl  = sec.get('topic_label', '')

            # Intro text (before first subsection)
            intro = (sec.get('intro_text') or '').strip()
            if intro:
                meta = {'chapter': chapter, 'section': sec_heading, 'subsection': ''}
                new_chunks = self._split_with_overlap(
                    intro, source_id, meta,
                    sec_heading, '', chapter, source_file,
                    sec_topic_id, sec_topic_lbl,
                    global_idx, 0,
                )
                all_chunks.extend(new_chunks)
                global_idx += len(new_chunks)

            # Each subsection
            for sub in sec.get('subsections', []):
                sub_text  = (sub.get('text') or '').strip()
                sub_head  = sub['heading']
                sub_tid   = sub.get('topic_id',    sec_topic_id)
                sub_tlbl  = sub.get('topic_label', sec_topic_lbl)
                if not sub_text:
                    continue
                meta = {'chapter': chapter, 'section': sec_heading, 'subsection': sub_head}
                new_chunks = self._split_with_overlap(
                    sub_text, source_id, meta,
                    sec_heading, sub_head, chapter, source_file,
                    sub_tid, sub_tlbl,
                    global_idx, 0,
                )
                all_chunks.extend(new_chunks)
                global_idx += len(new_chunks)

        # Fix total_chunks
        total = len(all_chunks)
        for c in all_chunks:
            c.total_chunks = total

        return all_chunks

    def chunk_all(self, structures: List[Dict]) -> List[MDChunk]:
        out: List[MDChunk] = []
        for s in structures:
            chunks = self.chunk_structure(s)
            out.extend(chunks)
        return out


# ── Run chunker ───────────────────────────────────────────────────────────
chunker    = MarkdownChunker(chunk_size_tokens=CHUNK_SIZE_TOKENS, overlap_percent=OVERLAP_PERCENT)
all_chunks = chunker.chunk_all(all_structures)

n_with_sub = sum(1 for c in all_chunks if c.subsection_name)
lengths    = [len(c.text) for c in all_chunks]
sections_seen = {}
for c in all_chunks:
    k = c.section_name or '(fallback)'
    sections_seen.setdefault(k, 0)
    sections_seen[k] += 1

print(f"Total chunks      : {len(all_chunks):,}")
print(f"Chunk size target : {chunker.chunk_size:,} chars (~{CHUNK_SIZE_TOKENS} tokens)")
print(f"Overlap           : {chunker.overlap:,} chars ({OVERLAP_PERCENT}%)")
print(f"With subsection   : {n_with_sub:,}")
print(f"Avg / min / max   : {sum(lengths)/len(lengths):,.0f} / {min(lengths):,} / {max(lengths):,} chars")
print(f"\nTop sections by chunk count:")
for k, v in sorted(sections_seen.items(), key=lambda x: -x[1])[:12]:
    print(f"  {v:4d}  {k}")


## Step 6 — Generate e5-base-v2 embeddings

`intfloat/e5-base-v2` requires an instruction prefix:  
- **Documents (passages)** → `"passage: {text}"`  
- **Queries** → `"query: {text}"`


In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

e5_model = SentenceTransformer(EMBED_MODEL, device=device)
test_dim = len(e5_model.encode("passage: hello world", normalize_embeddings=True))
print(f"Model : {EMBED_MODEL}   dim = {test_dim}")


In [ ]:
def embed_passages(texts: List[str], batch_size: int = EMBED_BATCH) -> np.ndarray:
    """Embed document texts with 'passage: ' prefix (e5 instruction)."""
    prefixed = [f"passage: {t}" for t in texts]
    vecs = e5_model.encode(
        prefixed,
        batch_size=batch_size,
        show_progress_bar=True,
        normalize_embeddings=True,
        convert_to_numpy=True,
    )
    return vecs.astype("float32")


def embed_query(text: str) -> np.ndarray:
    """Embed a query string with 'query: ' prefix (e5 instruction)."""
    v = e5_model.encode(
        f"query: {text}",
        normalize_embeddings=True,
        convert_to_numpy=True,
    )
    return v.astype("float32")


print("Embedding", len(all_chunks), "chunks …")
chunk_texts      = [c.text for c in all_chunks]
embeddings_array = embed_passages(chunk_texts)

print(f"\nEmbeddings shape : {embeddings_array.shape}")
norms = np.linalg.norm(embeddings_array, axis=1)
print(f"Norm mean/min/max: {norms.mean():.4f} / {norms.min():.4f} / {norms.max():.4f}")


## Step 7 — Build FAISS vector index

In [ ]:
import faiss

dim   = embeddings_array.shape[1]
index = faiss.IndexFlatIP(dim)   # inner product = cosine for L2-normalised vectors
index.add(embeddings_array)

faiss.write_index(index, FAISS_INDEX_PATH)
print(f"FAISS index built: {index.ntotal:,} vectors  dim={dim}")
print(f"Saved to          : {FAISS_INDEX_PATH}")

# Save chunk metadata (parallel to FAISS index)
chunk_meta_list = [
    {
        'chunk_id':        c.chunk_id,
        'source_document_id': c.source_document_id,
        'chapter':         c.chapter,
        'section_name':    c.section_name,
        'subsection_name': c.subsection_name,
        'topic_id':        c.topic_id,
        'topic_label':     c.topic_label,
        'source_file':     c.source_file,
        'chunk_index':     c.chunk_index,
        'total_chunks':    c.total_chunks,
        'start_char':      c.start_char,
        'end_char':        c.end_char,
        'text':            c.text,
    }
    for c in all_chunks
]

with open(CHUNK_META_JSON, 'w', encoding='utf-8') as fh:
    json.dump(chunk_meta_list, fh, ensure_ascii=False, indent=2)

print(f"Chunk metadata   : {CHUNK_META_JSON}  ({CHUNK_META_JSON.stat().st_size/1024:.1f} KB)")


In [ ]:
# Optional: export embeddings as JSON (large — skip if disk space is limited)
SAVE_EMBEDDINGS_JSON = False   # set True to enable

if SAVE_EMBEDDINGS_JSON:
    export_embed = [
        {**chunk_meta_list[i], 'embedding': embeddings_array[i].tolist()}
        for i in range(len(chunk_meta_list))
    ]
    with open(CHUNKS_JSON, 'w', encoding='utf-8') as fh:
        json.dump(export_embed, fh, ensure_ascii=False, default=str)
    size_mb = CHUNKS_JSON.stat().st_size / 1_048_576
    print(f"Saved embedded chunks → {CHUNKS_JSON}  ({size_mb:.1f} MB)")
else:
    print("Embedding JSON export skipped (SAVE_EMBEDDINGS_JSON=False).")


## Step 8 — RAG retrieval demo

In [ ]:
def retrieve(query: str, top_k: int = 5, topic_filter: Optional[int] = None) -> List[Dict]:
    """
    Retrieve the top-k most relevant chunks for a query.

    Args:
        query        : Natural language question.
        top_k        : Number of results to return.
        topic_filter : If set, restrict results to chunks with this topic_id.
    Returns:
        List of dicts with keys: score, chapter, section_name, subsection_name,
                                 topic_label, text, source_file.
    """
    q_vec = embed_query(query).reshape(1, -1)

    # Retrieve more candidates when filtering by topic
    fetch_k    = top_k * 10 if topic_filter is not None else top_k
    scores, ids = index.search(q_vec, fetch_k)

    results = []
    for score, idx_i in zip(scores[0], ids[0]):
        if idx_i < 0:
            continue
        meta = chunk_meta_list[idx_i]
        if topic_filter is not None and meta['topic_id'] != topic_filter:
            continue
        results.append({
            'score':           float(score),
            'chapter':         meta['chapter'],
            'section_name':    meta['section_name'],
            'subsection_name': meta['subsection_name'],
            'topic_id':        meta['topic_id'],
            'topic_label':     meta['topic_label'],
            'source_file':     meta['source_file'],
            'text':            meta['text'],
        })
        if len(results) >= top_k:
            break

    return results


def display_results(query: str, results: List[Dict]) -> None:
    """Pretty-print retrieval results."""
    print(f"\nQuery: {query!r}")
    print("=" * 72)
    for rank, r in enumerate(results, 1):
        sub = f" › {r['subsection_name']}" if r['subsection_name'] else ""
        print(f"#{rank}  score={r['score']:.4f}  [{r['chapter']}]")
        print(f"    {r['section_name']}{sub}")
        print(f"    topic: {r['topic_label']!r}")
        snippet = r['text'][:300].replace('\n', ' ')
        if len(r['text']) > 300:
            snippet += ' …'
        print(f"    {snippet}")
        print()


print("retrieve() and display_results() ready.")


In [ ]:
# ── Demo queries ─────────────────────────────────────────────────────────
queries = [
    "What are the trade-offs between consistency and availability in distributed systems?",
    "How do B-trees differ from LSM-trees for storage?",
    "What is the difference between OLTP and OLAP systems?",
    "How does scalability relate to load parameters?",
    "What are the nonfunctional requirements for data systems?",
]

for q in queries:
    hits = retrieve(q, top_k=3)
    display_results(q, hits)
    print("-" * 72)


### Topic-filtered retrieval

In [ ]:
# Show available topics
print("Available topics:")
for _, row in topic_info[topic_info.Name != '-1_'].iterrows():
    print(f"  topic {row['Topic']:3d}  |  count={row['Count']:3d}  |  {row['Name']}")


In [ ]:
# ── Change TOPIC_ID_FILTER to any topic from the table above ─────────────
TOPIC_ID_FILTER = 0   # <-- set to desired topic_id

q = "How does this system handle failures and recovery?"
hits = retrieve(q, top_k=5, topic_filter=TOPIC_ID_FILTER)
display_results(q, hits)


## Pipeline summary

In [ ]:
from collections import Counter

section_counts = Counter(c.section_name or '(fallback)' for c in all_chunks)
topic_counts   = Counter(c.topic_label                  for c in all_chunks)

print("=" * 72)
print("BOOK MARKDOWN RAG PIPELINE — SUMMARY")
print("=" * 72)
print(f"  Source files        : {len(md_files)}")
print(f"  Chapters            : {len(all_structures)}")
print(f"  H2 sections         : {total_sections}")
print(f"  H3 subsections      : {total_subsections}")
print(f"  Topics discovered   : {int(topic_info[topic_info.Name != '-1_'].shape[0])}")
print(f"  Structure JSON      : {STRUCTURE_JSON}")
print()
print(f"  Chunk size          : ~{CHUNK_SIZE_TOKENS} tokens / {chunker.chunk_size} chars")
print(f"  Overlap             : {OVERLAP_PERCENT}%")
print(f"  Total chunks        : {len(all_chunks):,}")
print(f"  Embed model         : {EMBED_MODEL}  (dim={test_dim})")
print(f"  FAISS index         : {FAISS_INDEX_PATH}  ({index.ntotal:,} vectors)")
print(f"  Chunk metadata JSON : {CHUNK_META_JSON}")
print()
print("  Top sections by chunk count:")
for sec, cnt in section_counts.most_common(8):
    print(f"    {cnt:4d}  {sec}")
print()
print("  Top topics by chunk count:")
for lbl, cnt in topic_counts.most_common(8):
    print(f"    {cnt:4d}  {lbl}")
print()
print("  Done. ✅")
